# mafutils benchmark summary

Wall-clock time, CPU, and memory for `index`/`validate`/`gc`/`stats`/`fetch`, 
across `none`/`gz`/`bgzip` compression, generated from `results/*.benchmark.tsv` 
(Snakemake's `benchmark:` directive -- see `../DEVELOPMENT.md` and `README.md`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

sys.path.insert(0, str(Path.cwd()))
from parse_benchmarks import load_benchmarks

with open("config.yaml") as fp:
    CONFIG = yaml.safe_load(fp)

COMPRESSION_ORDER = list(CONFIG["compressions"].keys())
PARALLEL_COMPRESSIONS = CONFIG["parallel_compressions"]
HEADLINE_P = CONFIG["headline_processes"]

## Load all benchmark TSVs

Each `results/{compression}/{command}[.headline|.sweep.p{N}].benchmark.tsv` is a 
standard Snakemake benchmark file (`s`, `max_rss`, `mean_load`, ...). File names 
encode compression/command/kind/processes, so no separate metadata store is needed. 
(Parsing lives in `parse_benchmarks.py`, shared with the Snakefile's `record_history` 
rule so the two can't drift out of sync.)

In [ ]:
bench = load_benchmarks("results", HEADLINE_P)
bench["compression"] = pd.Categorical(bench["compression"], categories=COMPRESSION_ORDER, ordered=True)
bench.head()

## Headline comparison

One run per command per compression, at `--processes 8` (matching 
`baseline-manual-timings.txt`) for the parallel-eligible commands.

In [ ]:
headline = bench[bench["kind"].isin(["single", "headline"])].copy()
headline_table = (
    headline.set_index(["command", "compression"])[["s", "max_rss", "mean_load"]]
    .rename(columns={"s": "wall_seconds", "max_rss": "max_rss_mb", "mean_load": "mean_cpu_pct"})
    .sort_index()
)
headline_table

In [ ]:
COMMAND_ORDER = ["index", "validate", "gc", "stats", "fetch"]
pivot_time = headline.pivot(index="command", columns="compression", values="s").reindex(COMMAND_ORDER)

ax = pivot_time.plot(kind="bar", logy=True, figsize=(8, 5))
ax.set_ylabel("wall-clock time (s, log scale)")
ax.set_xlabel("")
ax.set_title("Headline wall-clock time by command and compression")
ax.legend(title="compression")
plt.tight_layout()
plt.show()

In [ ]:
pivot_mem = headline.pivot(index="command", columns="compression", values="max_rss").reindex(COMMAND_ORDER)

ax = pivot_mem.plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("max RSS (MB)")
ax.set_xlabel("")
ax.set_title("Headline peak memory by command and compression")
ax.legend(title="compression")
plt.tight_layout()
plt.show()

## Process-count scaling (`gc` / `stats` / `fetch`)

`none` and `bgzip` get real random access, so `--processes` should show real 
speedup. `gz` cannot be parallelized (each worker would have to redundantly 
re-decompress everything before its own start point -- see `DEVELOPMENT.md`), so 
it always falls back to a single sequential pass regardless of `--processes`; 
rather than re-measure that known-constant behavior at every process count, its 
one data point is drawn as a flat reference line. For `gc`/`stats`, that's the 
headline run (`--processes 8`, full indexed MAF) -- directly comparable, since 
`gc`/`stats`'s headline and sweep runs both process the *entire* MAF regardless of 
`--processes`. `fetch` needs its own gz reference instead of reusing its headline: 
`fetch`'s sweep intentionally uses a much smaller BED subset than its headline run 
(`fetch_sweep_regions` in `config.yaml`, to keep the sweep matrix fast -- see 
`README.md`), so gz's *headline* time (full BED) isn't comparable to none/bgzip's 
*sweep* times (5000-region subset) -- not even after normalizing by region count, 
since the smaller sweep's per-region time is more dominated by fixed per-run overhead 
(index parsing, MAF header, species-file loading) than the full run's steady-state 
per-region cost. `fetch_sweep_gz` in the Snakefile runs gz on the *same* 5000-region 
sweep BED instead, giving a real, directly comparable number.

In [ ]:
sweep = bench[bench["kind"] == "sweep"].copy()
gz_headline = headline[headline["compression"] == "gz"].set_index("command")["s"]
gz_sweep_ref = sweep[(sweep["compression"] == "gz") & (sweep["command"] == "fetch")]["s"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
for ax, command in zip(axes, ["gc", "stats", "fetch"]):
    cmd_sweep = sweep[sweep["command"] == command]
    for compression in PARALLEL_COMPRESSIONS:
        line = cmd_sweep[cmd_sweep["compression"] == compression].sort_values("processes")
        ax.plot(line["processes"], line["s"], marker="o", label=compression)
    if command == "fetch" and not gz_sweep_ref.empty:
        ax.axhline(
            gz_sweep_ref.iloc[0], linestyle="--", color="gray",
            label="gz (same 5000-region subset, --processes ignored)",
        )
    elif command in gz_headline.index:
        ax.axhline(gz_headline[command], linestyle="--", color="gray", label="gz (headline, --processes ignored)")
    ax.set_title(command)
    ax.set_xlabel("--processes")
    ax.set_ylabel("wall-clock time (s)")

# One shared legend below all three panels instead of one per panel --
# per-panel legends were overlapping the plotted lines/markers.
handles_by_label = {}
for ax in axes:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        handles_by_label[label] = handle
fig.legend(handles_by_label.values(), handles_by_label.keys(), loc="lower center", ncol=len(handles_by_label), bbox_to_anchor=(0.5, -0.08))
fig.suptitle("Process-count scaling by compression")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## CPU utilization

`mean_cpu_pct` above ~100% per core requested indicates the run was genuinely 
using multiple cores; a flat, low value close to 100% total regardless of 
`--processes` indicates the run is I/O-bound rather than CPU-bound (e.g. waiting 
on the filesystem), so throwing more processes at it wouldn't help further.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
for ax, command in zip(axes, ["gc", "stats", "fetch"]):
    cmd_sweep = sweep[sweep["command"] == command]
    for compression in PARALLEL_COMPRESSIONS:
        line = cmd_sweep[cmd_sweep["compression"] == compression].sort_values("processes")
        ax.plot(line["processes"], line["mean_load"], marker="o", label=compression)
    ax.set_title(command)
    ax.set_xlabel("--processes")
axes[0].set_ylabel("mean CPU load (%)")
axes[0].legend()
fig.suptitle("CPU utilization by compression and process count")
plt.tight_layout()
plt.show()

## Compression cost, relative to the fastest variant per command

In [ ]:
relative = pivot_time.div(pivot_time.min(axis=1), axis=0).round(2)
relative

## Historical trend

`history.csv` gets one appended row per (command, compression, kind, processes) 
every time this pipeline is run to completion (see the Snakefile's `record_history` 
rule), tagged with the git commit and a timestamp -- unlike `results/`, it's meant 
to be committed and grow over time, so performance can be tracked across code changes 
rather than only compared within a single run.

In [ ]:
history_path = Path("history.csv")
if not history_path.exists():
    print("No history.csv yet -- this appears to be the first run.")
else:
    history = pd.read_csv(history_path, parse_dates=["run_timestamp"])
    history = history[history["kind"].isin(["single", "headline"])].copy()
    history = history.sort_values("run_timestamp")
    # One x-axis label per distinct run (timestamp + commit), in chronological order.
    run_order = history[["run_timestamp", "git_commit"]].drop_duplicates().reset_index(drop=True)
    run_order["run_label"] = (
        run_order["run_timestamp"].dt.strftime("%Y-%m-%d %H:%M") + "\n" + run_order["git_commit"]
    )
    history = history.merge(run_order, on=["run_timestamp", "git_commit"])

    if len(run_order) < 2:
        print(f"Only {len(run_order)} recorded run(s) so far -- trend charts need at least 2.")
    else:
        fig, axes = plt.subplots(len(COMMAND_ORDER), 1, figsize=(9, 3 * len(COMMAND_ORDER)), sharex=True)
        for ax, command in zip(axes, COMMAND_ORDER):
            cmd_history = history[history["command"] == command]
            for compression in COMPRESSION_ORDER:
                line = cmd_history[cmd_history["compression"] == compression].sort_values("run_timestamp")
                if not line.empty:
                    ax.plot(line["run_label"], line["s"], marker="o", label=compression)
            ax.set_ylabel("wall-clock time (s)")
            ax.set_title(command)
            ax.legend(fontsize="small")
        axes[-1].set_xlabel("run (timestamp / git commit)")
        fig.suptitle("Headline wall-clock time across recorded runs")
        plt.tight_layout()
        plt.show()